# Can we predict late delivery?
Protocol: `protocol.yaml` (temporal holdout from `holdout.csv`, 5-split walk-forward CV on training rows, choose by mean CV ROC AUC, score the holdout once).

**Feature decision.** `claim_paid_eur` is compensation paid on a claim *settled for the consignment*, i.e. recorded after delivery, so it is not known at prediction time (dispatch). It is almost a copy of the target (claim > 0 -> ~91% late; claim = 0 -> 0% late). It is excluded. `consignment` (identifier) and `dispatched_on` (split column) are excluded per the protocol.

In [ ]:
import json, hashlib
import numpy as np, pandas as pd
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

assert hashlib.sha256(open('data.csv','rb').read()).hexdigest()=='859a474044a292565d72b06a6180ade1c1adbc8c72a47c4a518186876f581f27'
assert hashlib.sha256(open('holdout.csv','rb').read()).hexdigest()=='fae5c02170970fe92fcadaee6d1109bb1d459c9dc033dfc487bb2c823d0f6736'
df=pd.read_csv('data.csv'); df['dispatched_on']=pd.to_datetime(df['dispatched_on'])
hold_rows=pd.read_csv('holdout.csv')['row'].to_numpy()
assert len(df)==8051 and len(hold_rows)==1610
train_rows=np.setdiff1d(np.arange(len(df)),hold_rows)
# order training rows by time for walk-forward CV
train_rows=train_rows[np.argsort(df['dispatched_on'].to_numpy()[train_rows],kind='stable')]
TARGET='late_delivery'
CAT=['carrier']; NUM=['distance_km','weight_kg','parcels']
FEATURES=CAT+NUM
X=df[FEATURES]; y=df[TARGET]
print(len(train_rows),len(hold_rows), y.iloc[train_rows].mean(), y.iloc[hold_rows].mean())

## Candidates (named before any scoring)

In [ ]:
def prep():
    return ColumnTransformer([('c',OneHotEncoder(handle_unknown='ignore'),CAT),('n',StandardScaler(),NUM)])
CANDIDATES={
 'baseline_prior': DummyClassifier(strategy='prior'),
 'logreg_C0.1': Pipeline([('p',prep()),('m',LogisticRegression(C=0.1,max_iter=1000))]),
 'logreg_C1': Pipeline([('p',prep()),('m',LogisticRegression(C=1.0,max_iter=1000))]),
 'hgb_depth3': Pipeline([('p',prep()),('m',HistGradientBoostingClassifier(max_depth=3,learning_rate=0.05,max_iter=200,random_state=0))]),
 'hgb_depth6': Pipeline([('p',prep()),('m',HistGradientBoostingClassifier(max_depth=6,learning_rate=0.05,max_iter=200,random_state=0))]),
}
list(CANDIDATES)

## Walk-forward CV on training rows (preprocessing is inside each pipeline, refit per fold)

In [ ]:
tscv=TimeSeriesSplit(n_splits=5)
folds=[(train_rows[a],train_rows[b]) for a,b in tscv.split(train_rows)]
cv_scores={}
for name,m in CANDIDATES.items():
    s=[]
    for tr,te in folds:
        mm=clone(m).fit(X.iloc[tr],y.iloc[tr])
        s.append(float(roc_auc_score(y.iloc[te],mm.predict_proba(X.iloc[te])[:,1])))
    cv_scores[name]=s
cv=pd.DataFrame(cv_scores).T; cv['mean']=cv.mean(axis=1); cv.sort_values('mean',ascending=False)

## Choose by highest mean CV ROC AUC, refit on all training rows, score holdout once

In [ ]:
chosen=max(cv_scores,key=lambda k:np.mean(cv_scores[k]))
final=clone(CANDIDATES[chosen]).fit(X.iloc[train_rows],y.iloc[train_rows])
holdout_auc=float(roc_auc_score(y.iloc[hold_rows],final.predict_proba(X.iloc[hold_rows])[:,1]))
print(chosen,'holdout roc_auc =',round(holdout_auc,4))

In [ ]:
evidence={
 'features':FEATURES,
 'candidates':list(CANDIDATES),
 'cv_folds':[[tr.tolist(),te.tolist()] for tr,te in folds],
 'cv_scores':cv_scores,
 'chosen':chosen,
 'holdout_rows':hold_rows.tolist(),
 'holdout_scores':{chosen:holdout_auc},
}
json.dump(evidence,open('evidence.json','w'))

## Conclusion
See the CV table and holdout ROC AUC above. Only dispatch-time features (carrier, distance, weight, parcels) are used; the post-outcome `claim_paid_eur` was excluded as leakage.